# 05 — Frame selection, Sun‑centric alignment, stacking

1. **Selection** (paper Section 2): drop frames whose CHT lunar radius lies outside
   mean ± 1σ → 130 of 136 polarimetric frames.  Writes `products/frame_table.csv`
   and the appendix metadata table.
2. **Alignment + stacking** (Section 3.2): for every (polariser, exposure) the
   synthetic‑luminance frames are translated (bilinear) so that the Sun centre
   from step 04 lands on the array centre `(4128, 2752)`, then **averaged**.
   Output: `products/stacked/stacked_exp{N}.fits`, 3 planes = pol1, pol2, pol3.

The stacking step was originally run by C. Gandhi off‑line (translate to the Sun
centre, arithmetic mean); it is re‑implemented here and validated against his
files in the last cell.  Two things to know:

* the manuscript says *median* — the data and the author say **mean**;
* his stacks left out **six more frames** than the 1‑σ filter
  (`config.EXTRA_EXCLUDED_FRAMES`, recovered by fitting his stacks onto the
  candidate frames), so the paper's "130 frames" and the per‑column counts in
  the histogram figure describe the filter, not the stacks (124 frames).
  The `use` column of `products/frame_table.csv` is what is actually stacked.

⏱ reads all frames (~70 GB) — about 5 minutes on the SSD.

In [ ]:
import sys, time
sys.path.insert(0, "..")          # config.py / utils.py live one level up
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from astropy.io import fits

import config, utils
%matplotlib inline

In [ ]:
table = utils.build_frame_table(config.SUN_MOON_CENTERS_CSV)
print(f"lunar radius: mean {table.attrs['moon_radius_mean']:.2f} px, sigma {table.attrs['moon_radius_std']:.2f} px")
print(f"frames before filter: {len(table)}   after {config.RADIUS_FILTER_NSIGMA:.0f}-sigma filter: {table.radius_ok.sum()}   stacked: {table.use.sum()}")
table.to_csv(config.FRAME_TABLE_CSV, index=False)
print("rejected by the radius filter:"); display(table[~table.radius_ok][["filename", "position", "inverse_exposure_time", "moon_radius"]])
print("additionally left out of the paper's stacks (config.EXTRA_EXCLUDED_FRAMES):")
table[~table.manual_ok][["filename", "position", "inverse_exposure_time", "moon_radius"]].assign(reason=lambda d: d.filename.map(config.EXTRA_EXCLUDED_FRAMES))

In [ ]:
good = table[table.use]
pd.crosstab(good.inverse_exposure_time, good.position).reindex(list(config.INV_EXPOSURES))

In [ ]:
# Appendix table (identical to data/paper_metadata_documentation.csv up to the order of equal-exposure rows)
meta = utils.paper_metadata_table(table)
meta.to_csv(config.PAPER_METADATA_CSV, index=False)
latex = meta.assign(Filename=meta.Filename.str.replace("_", r"\_")).to_latex(
    index=False, escape=False, longtable=True, column_format="p{4.5cm} c c c c", float_format="%.2f",
    caption=r"Summary of observational data and corresponding metadata for the calibrated eclipse frames. "
            r"Frames listed here passed the 1-$\sigma$ lunar radius variance filter and were subsequently "
            r"utilised in the HDR compositing stage.", label="tab:observation_metadata")
(config.PRODUCTS_DIR / "paper_metadata_table.tex").write_text(latex)
print(latex[:400], "...")

In [ ]:
from tqdm.auto import tqdm
centers = good.set_index("filename")
for inv_exp in tqdm(config.INV_EXPOSURES, desc="exposures"):
    planes, nframes = [], []
    for pos in config.POLARIZER_POSITIONS:
        files = list(good[(good.position == pos) & (good.inverse_exposure_time == inv_exp)].filename)
        planes.append(utils.stack_frames(files, centers))
        nframes.append(len(files))
    cube = np.stack(planes, axis=0).astype(np.float32)
    hdr = utils.read_header(files[0])
    hdr["EXPTIME"] = 1.0 / inv_exp
    hdr["FILT-1"] = "pol1,pol2,pol3"
    hdr["HISTORY"] = "Paper-I-Analysis 05: luminance, shift to Sun centre, mean"
    hdr["SUNROT"] = (config.SUN_CENTER_ROTATION, "Sun-centre rotation convention")
    for i, pos in enumerate(config.POLARIZER_POSITIONS):
        hdr[f"NFRAME{pos}"] = (nframes[i], f"frames averaged in plane {i} (pol{pos})")
    hdr["SUNCX"], hdr["SUNCY"] = config.SUN_CENTER_XY
    fits.writeto(utils.stacked_filename(inv_exp), cube, header=hdr, overwrite=True)
    print(f"1/{inv_exp:>3d} s  frames {nframes}  ->  {utils.stacked_filename(inv_exp).name}")

In [ ]:
fig, axes = plt.subplots(3, len(config.INV_EXPOSURES), figsize=(2.2*len(config.INV_EXPOSURES), 5))
for j, inv_exp in enumerate(config.INV_EXPOSURES):
    d = fits.getdata(utils.stacked_filename(inv_exp), memmap=True)
    for i in range(3):
        axes[i, j].imshow(utils.asinh_stretch(np.asarray(d[i, ::8, ::8], float)), cmap="gray"); axes[i, j].axis("off")
        if i == 0: axes[i, j].set_title(f"1/{inv_exp} s", fontsize=9)
    if j == 0:
        for i, pos in enumerate(config.POLARIZER_POSITIONS): axes[i, 0].text(-0.15, 0.5, f"pol{pos}", transform=axes[i, 0].transAxes, rotation=90, va="center")
plt.tight_layout()

In [ ]:
# Validation against the stacks that produced the paper (central 2500 x 3500 px window).
# In "legacy" mode every plane matches at corr >= 0.9997; in "astrometric" mode the
# stacks are re-registered by up to ~11 px, so the correlation drops slightly (expected).
if config.LEGACY_STACKED_DIR.exists():
    region = (slice(1500, 4000), slice(2500, 6000))
    rows = []
    for inv_exp in config.INV_EXPOSURES:
        ours = fits.getdata(utils.stacked_filename(inv_exp), memmap=True)
        legacy = fits.getdata(config.LEGACY_STACKED_DIR / f"Linear_Composite_{inv_exp}.fits", memmap=True)
        for i, pos in enumerate(config.POLARIZER_POSITIONS):
            rows.append(dict(inv_exp=inv_exp, pol=pos, **utils.compare_arrays(ours[i], legacy[i], region)))
    val = pd.DataFrame(rows)
    val.to_csv(config.PRODUCTS_DIR / "validation_stacked_vs_legacy.csv", index=False)
    print(f"mode = {config.SUN_CENTER_ROTATION};  min corr = {val['corr'].min():.5f}")
    display(val.pivot(index="inv_exp", columns="pol", values="corr").round(5))
else:
    print("legacy stacks not found — skipping")